# PointVisor – Point-Supervised Semantic Segmentation on DLRSD
**LandVisor Project Task Solution**

Implements partial Focal CE loss, simulates point annotations on real remote sensing data, trains a segmentation model, and runs experiments.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import random
from glob import glob
from tqdm import tqdm
import pandas as pd

DATA_ROOT = "DLRSD"
IMAGE_DIR = os.path.join(DATA_ROOT, "Images")
LABEL_DIR = os.path.join(DATA_ROOT, "Labels")

Images found: 0


In [8]:
# Verify paths
print("DATA_ROOT  :", DATA_ROOT)
print("IMAGE_DIR  :", IMAGE_DIR)
print("LABEL_DIR  :", LABEL_DIR)

# Define all possible image extensions
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff']
all_images = []

for ext in exts:
    # Search for both lowercase and uppercase versions
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    all_images.extend(glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))

print(f"Total images found: {len(all_images)}")

if len(all_images) > 0:
    print("First 5 image paths:")
    for p in all_images[:5]:
        print("   ", p)
else:
    # If still 0, check what is actually in the directory
    print(f"\n[!] ALERT: No images found in {IMAGE_DIR}")
    print("Actual directory contents:", os.listdir(IMAGE_DIR)[:10])

DATA_ROOT  : DLRSD
IMAGE_DIR  : DLRSD\Images
LABEL_DIR  : DLRSD\Labels
Total images found: 4200
First 5 image paths:
    DLRSD\Images\agricultural\agricultural00.tif
    DLRSD\Images\agricultural\agricultural01.tif
    DLRSD\Images\agricultural\agricultural02.tif
    DLRSD\Images\agricultural\agricultural03.tif
    DLRSD\Images\agricultural\agricultural04.tif


In [ ]:
class PartialFocalLoss(nn.Module):
    """Partial Focal CE Loss"""
    def __init__(self, gamma=2.0, alpha=0.25, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, pred, target, mask):
        # pred: (B, C, H, W) logits
        # target: (B, H, W) long
        # mask: (B, H, W) float 0/1 (only labeled points = 1)
        ce = nn.functional.cross_entropy(pred, target, reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        masked = focal * mask
        return masked.sum() / (mask.sum() + 1e-8)   # average only over labeled points

In [ ]:
class DLRSDPointDataset(Dataset):
    def __init__(self, image_files, points_per_class=5):
        self.image_files = image_files
        self.points_per_class = points_per_class

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
    
        # Safer label path (works with subfolders)
        rel_path = os.path.relpath(img_path, IMAGE_DIR)
        label_path = os.path.join(LABEL_DIR, rel_path).replace('.jpg', '.png').replace('.JPG', '.png')

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)  # 0-16 classes
        label = label.astype(np.int64)

        # Simulate point labels
        point_target, point_mask = self.simulate_points(label)

        return torch.from_numpy(img), torch.from_numpy(label), point_target, point_mask

    def simulate_points(self, label_np):
        H, W = label_np.shape
        point_target = np.full((H, W), 255, dtype=np.int64)
        point_mask = np.zeros((H, W), dtype=np.float32)

        for c in range(17):  # 17 classes in DLRSD
            ys, xs = np.where(label_np == c)
            if len(ys) == 0: continue
            n = min(self.points_per_class, len(ys))
            idx = random.sample(range(len(ys)), n)
            point_target[ys[idx], xs[idx]] = c
            point_mask[ys[idx], xs[idx]] = 1.0

        return point_target, point_mask

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',
                 in_channels=3, classes=17).to(device)

criterion = PartialFocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(dataloader, points_per_class):
    model.train()
    total_loss = 0
    for img, full_label, point_target, point_mask in tqdm(dataloader):
        img, point_target, point_mask = img.to(device), point_target.to(device), point_mask.to(device)
        
        optimizer.zero_grad()
        pred = model(img)
        loss = criterion(pred, point_target, point_mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [ ]:
# Use only first 300 images for fast experiments, it can be increasd
# Robust file collection for DLRSD subfolder structure
image_files = sorted(
    glob(os.path.join(IMAGE_DIR, "**/*.jpg"), recursive=True) +
    glob(os.path.join(IMAGE_DIR, "**/*.png"), recursive=True)
)

print(f"Total images found: {len(image_files)}")

# Use first 300 for fast experiments (change to 1000+ when ready)
all_images = image_files[:300]

results = []

for n_points in [5, 15, 30]:
    for use_focal in [True, False]:
        print(f"\n=== Experiment: {n_points} points per class, Focal={use_focal} ===")
        
        dataset = DLRSDPointDataset(all_images, points_per_class=n_points)
        loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)  # num_workers=0 to avoid Windows issues
        
        criterion = PartialFocalLoss(gamma=2.0) if use_focal else nn.CrossEntropyLoss(ignore_index=255)
        
        for epoch in range(5):   # 5 epochs for demo (increase later)
            loss = train_one_epoch(loader, n_points)
            print(f"  Epoch {epoch+1}/5 - Loss: {loss:.4f}")
        
        results.append({
            "points_per_class": n_points,
            "focal": use_focal,
            "final_loss": loss,
        })

df = pd.DataFrame(results)
print("\n=== EXPERIMENT RESULTS ===")
print(df)
df.to_csv("experiment_results.csv", index=False)

Run the cell below to see an example image + points